# Drone Chase Tag: Elo turnuvası (GPU)

Bir self-play koşusunun bütün sürümleri arasında round-robin: her kovalayan × her kaçan, çift başına 256 maç,
görülmemiş arenalar. Koşunun dosyaları (`experiments/<koşu>/`) repoda olduğu için eğitim gerekmez.

1. **Çalışma zamanı → Çalışma zamanı türünü değiştir → A100 GPU** (yoksa L4) seç.
2. Hücreleri sırayla çalıştır. Sonuç Drive'a (`results/<koşu>/elo/`) yazılır ve zip olarak iner.
3. Zip'i `D:\İndirilenler`e indir; `experiments/<koşu>/elo/` altına konup commit'lenir.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
os.chdir("/content")                        # hücre nerede çalışırsa çalışsın, önce sabit bir yere geç
REPO = "https://github.com/HasaNSaygili/DroneChaseTag.git"
if os.path.isdir("/content/proje/.git"):    # daha önce indirildiyse silmeden güncelle
    !git -C /content/proje fetch -q origin && git -C /content/proje reset -q --hard origin/main
else:
    !git clone -q {REPO} /content/proje
assert os.path.isfile("/content/proje/scripts/elo.py"), "Kod indirilemedi: bu hücrenin çıktısını sohbete yapıştır"
os.chdir("/content/proje")
!git log --oneline -1
!pip install -q "crazyflow[gpu]==0.4.0" -e .

In [ ]:
# Bütün koşular burada kayıtlı: Elo'su alınacak self-play koşusu (experiments/ altındaki klasör adı).
# RUN'a çalıştırmak istediğinin adını yaz. GAMES: çift başına maç. DET: True = gürültüsüz (varsayılan olasılıklı).
RUNS = ["sp-faz1-evstd-s0"]
RUN = "sp-faz1-evstd-s0"
RUN = RUN.removeprefix("elo-")   # runs/ altındaki notebook'un adı "elo-<koşu>"
GAMES = 256
DET = False
OUT = "/content/drive/MyDrive/DroneChaseTag/results"
assert os.path.isdir(f"/content/proje/experiments/{RUN}"), f"experiments/{RUN} repoda yok"
print(RUN, "| çift başına", GAMES, "maç |", "gürültüsüz" if DET else "olasılıklı")

In [ ]:
import os
os.chdir("/content/proje")
!python -W ignore scripts/elo.py --run experiments/{RUN} --games {GAMES} {'--deterministic' if DET else ''} 2>&1 | grep -v -E "warp|spec|fromstring"

In [ ]:
# Sonuç: grafiği göster, Drive'a kopyala, zip olarak indir
import shutil, zipfile
from IPython.display import Image, display
from google.colab import files
sub = "elo_det" if DET else "elo"
src = f"/content/proje/experiments/{RUN}/{sub}"
display(Image(f"{src}/elo.png"))
shutil.copytree(src, f"{OUT}/{RUN}/{sub}", dirs_exist_ok=True)
zpath = f"/content/{RUN}_{sub}.zip"
with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
    for name in os.listdir(src):
        z.write(f"{src}/{name}", f"{RUN}/{sub}/{name}")
print(zpath)
files.download(zpath)